# GRPO, RLHF, and Multi‑Reward Training (Math • Code • General)

This notebook demonstrates **RLHF‑style fine‑tuning using GRPO** with **three reward models**:

- **Math Reward — correctness**
- **Code Reward — structural correctness**
- **General Reward — DeBERTa preference model**

## RLHF — Intuition

RLHF = **Reinforcement Learning from Human Feedback**:

1. A base LLM generates multiple outputs.
2. Humans (or a reward model trained from humans) judge which ones are better.
3. The model is fine‑tuned to prefer the better responses.

So instead of learning *"predict the next token"*, the model learns:

`"Generate the answer that people like most."`

## GRPO — Intuition

**Group‑Relative Policy Optimization** works like PPO but compares **multiple completions for the same prompt**.

- Sample *N* responses per prompt
- Score each with one or more reward functions
- Push the policy toward the *best in the group*

This is powerful because rewards don't have to be perfect — they just need to **rank answers reliably**.

We also follow the **DeepSeek‑style multi‑reward idea**:

- Math questions → Math reward only
- Coding questions → Code reward only
- General questions → DeBERTa reward only

## 1. Training & Model Configuration

We configure GRPO to sample multiple completions per prompt and compute rewards.

In [ ]:
import torch
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig, ModelConfig, ScriptArguments
from trl.rewards import accuracy_reward

# Base model config
model_args = ModelConfig(
    model_name_or_path="Qwen/Qwen2-0.5B-Instruct",   # Small but capable model
    torch_dtype="bfloat16" if torch.cuda.is_available() else "float32",
)

# GRPO training hyperparameters
training_args = GRPOConfig(
    output_dir="grpo-test-output",
    num_train_epochs=3,
    per_device_train_batch_size=2,   # 2 prompts per batch step
    num_generations=8,               # each prompt → 8 completions
    generation_batch_size=16,        # 2 × 8 = 16 total generations per step
    max_completion_length=128,
    learning_rate=5e-6,
    beta=0.04,
    report_to="none",
    logging_steps=1,
)

# Simple dataset with task type + solution for math
dataset = Dataset.from_list([
    {"prompt": "What is 2+2?", "task": "math", "solution": "4"},
    {"prompt": "Write a function that returns the sum of two numbers.", "task": "code", "solution": ""},
    {"prompt": "What is 3*4?", "task": "math", "solution": "12"},
    {"prompt": "Write a function that returns the product of two numbers.", "task": "code", "solution": ""},
    {"prompt": "Explain in simple terms why the sky appears blue during the day.", "task": "general", "solution": ""},
    {"prompt": "Give three practical tips to stay focused while studying for exams.", "task": "general", "solution": ""},
])

## 2. Reward Model: General Language Quality (DeBERTa)

This reward fires **only when `task == general`**.

We use the OpenAssistant DeBERTa‑v3 reward model to score

`prompt + answer → quality score`

In [ ]:
def deberta_reward_func(prompts, completions, task, **kwargs):
    """
    Reward for general QA quality using
    OpenAssistant/reward-model-deberta-v3-large-v2
    """
    from transformers import pipeline

    # Lazy‑load the model only once
    global deberta_rm
    if "deberta_rm" not in globals():
        deberta_rm = pipeline(
            "text-classification",
            model="OpenAssistant/reward-model-deberta-v3-large-v2",
            device=0,
        )

    rewards = []

    for p, c, t in zip(prompts, completions, task):
        if t != "general":
            rewards.append(None)
            continue

        # Handle chat‑style responses if needed
        if isinstance(c, list) and len(c) > 0 and isinstance(c[0], dict):
            text = c[0].get("content", str(c))
        else:
            text = str(c)

        pair_text = f"Question:\n{p}\n\nAnswer:\n{text}"
        out = deberta_rm(pair_text, truncation=True, max_length=512)
        rewards.append(float(out[0]["score"]))

    return rewards

## 3. Reward Model: Math Correctness

We wrap the output into the format expected by `accuracy_reward`.

This fires **only when `task == math`**.

In [ ]:
def math_reward_func(prompts, completions, task, solution, **kwargs):
    # Boolean mask for math rows
    mask = [t == "math" for t in task]

    # Extract only math examples
    math_completions = [c for c, m in zip(completions, mask) if m]
    math_solutions = [s for s, m in zip(solution, mask) if m]

    # Edge case — no math rows in this batch
    if not math_completions:
        return [None] * len(completions)

    # Wrap outputs for TRL helper
    math_completions_wrapped = [
        [{"role": "assistant", "content": f"My answer is \\boxed{{{c}}}"}] for c in math_completions
    ]

    # Compute math reward
    math_rewards = accuracy_reward(
        completions=math_completions_wrapped,
        solution=math_solutions,
    )

    # Reshape back to batch size
    it = iter(math_rewards)
    rewards = [next(it) if m else None for m in mask]
    return rewards

## 4. Reward Model: Code Structure Check

We reward responses that **look like a Python function**.

This fires **only when `task == code`**.

In [ ]:
def test_code_solution(prompt: str, completion: str) -> bool:
    text = completion.lower()
    # Minimal heuristic — in real life you'd run unit tests
    return ("def " in text) and ("return" in text)

def coding_reward_func(prompts, completions, task, **kwargs):
    rewards = []
    for p, c, t in zip(prompts, completions, task):
        if t == "code":
            works = test_code_solution(p, c)
            rewards.append(1.0 if works else -1.0)
        else:
            rewards.append(None)   # Not a coding sample
    return rewards

## 5. Sanity‑Check Reward Functions

Before training, we verify that rewards behave as expected:

In [ ]:
print("--- Testing Math Reward ---")
math_prompts = ["What is 2+2?", "What is simple form of 2 by 6?"]
math_tasks = ["math", "math"]
math_solutions = [r"4", r"\\frac{1}{3}"]
math_completions = [r"40", r"1/3"]

math_results = math_reward_func(math_prompts, math_completions, math_tasks, math_solutions)
print(f"Good Math Reward (Expected 0.0): {math_results[0]}")
print(f"Bad Math Reward (Expected 1.0): {math_results[1]}")

print("\n--- Testing Coding Reward ---")
code_prompts = ["Write a sum function", "Write a sum function"]
code_tasks = ["code", "code"]
code_completions = ["def add(a,b): return a+b", "I don't know how to code that"]
code_results = coding_reward_func(code_prompts, code_completions, code_tasks)
print(f"Good Code Reward (Expected 1.0): {code_results[0]}")
print(f"Bad Code Reward (Expected -1.0): {code_results[1]}")

print("\n--- Testing General Reward ---")
gen_prompts = ["Why is the sky blue?", "Why is the sky blue?"]
gen_tasks = ["general", "general"]
gen_completions = [
    "The sky appears blue because of Rayleigh scattering. Shorter blue wavelengths are scattered more by the atmosphere.",
    "Sky blue because magic and I hate questions."
]
gen_results = deberta_reward_func(gen_prompts, gen_completions, gen_tasks)
print(f"Good General Reward (Higher score): {gen_results[0]}")
print(f"Bad General Reward (Lower score): {gen_results[1]}")

## 6. Train with GRPO

We now pass **all reward functions** to the trainer.

In [ ]:
trainer = GRPOTrainer(
    model=model_args.model_name_or_path,
    reward_funcs=[math_reward_func, coding_reward_func, deberta_reward_func],
    train_dataset=dataset,
    args=training_args,
)

trainer.train()
trainer.save_model(training_args.output_dir)

## 7. Plot Training Loss

Plotting is kept **in its own block**, as requested 👍

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
steps = [x['step'] for x in history if 'loss' in x]
loss = [x['loss'] for x in history if 'loss' in x]

plt.figure(figsize=(10,5))
plt.plot(steps, loss, label='Training Loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('GRPO Training Loss')
plt.legend()
plt.grid(True)
plt.show()

# 8. Summary

## Summary — What We Built

In this notebook we walked through a **practical RLHF-style training pipeline** using **GRPO**.  
Instead of relying on a single reward model, we combined **three specialized rewards**:

- **Math reward** — checks correctness using `accuracy_reward`
- **Code reward** — simple structural validation (`def` + `return`)
- **General reward** — a DeBERTa-v3 preference model for natural-language quality

GRPO samples **multiple completions per prompt**, scores them, and **pushes the model toward the relatively better responses** within each group. This mirrors how modern LLMs are aligned: not just predicting text, but **optimizing for desirable behavior across different skill domains**.

This was a **small, educational setup**, but the structure is the same one used in large-scale RLHF systems — just with simpler rewards and a smaller model.
